In [1]:
##################################
# Ribosome tunnel model building #
##################################
# OVERALL IDEA
# - Each selected ribosome structure was first aligned to the 3D cryo-EM map of EMDB-14850.
# - The fit was further refined by rigid-body docking specifically into the isolated tunnel region map (tunnel_6A.mrc).
# - An initial atomistic model of the tunnel was generated using a custom VMD script (tunnel_generator.tcl) to extract all atoms within the tunnel_6A.mrc density.
# - Missing tunnel segments not captured by the initial map were manually reconstructed, and post-translational/post-transcriptional modifications incompatible with the SMOG all-atom AMBER force field were removed.
# - A model of the nascent chain (NC) was inserted, and steric clashes were resolved; extensive rebuilding of the NC was performed in Coot when necessary.
# - The final system was parameterised using the SMOG all-atom force field with AMBER bonded potentials. All native contacts within the NC itself, as well as between the NC and the ribosome, were deleted to allow unbiased exploration of the tunnel.
# - Molecular dynamics simulations (100 ns) were performed using GROMACS 2021.

In [10]:
# Fitting in chimeraX to the EMDB-14850 and tunnel_6A.mrc file

In [12]:
import numpy as np
import os

In [13]:
path = "Projects/ribosome_tunnels/bacteria/sfra/" # Path to the directory with ribosome structure 

In [14]:
os.chdir(path)

In [15]:
id = "9qt5" #PDB_id corresponding to the ribosome structure

In [16]:
def write_tcl_script(id_value):
    """Generates the VMD script to extract the tunnel based on cryo-EM density."""
    tcl_script = f"""set id "{id_value}"
# Loading files 
mol new {id_value}_fitted.pdb
mol addfile tunnel_6A.mrc
# Selecting region that corresponds to the tunnel
set a [atomselect top "vol0 > 0 and (protein or nucleic)"]
set sorted_i [lsort -uniq -integer [$a get residue]]
set extended_list []
set N [llength $sorted_i]
# Ensure there are no singletons (minimum 3 residues per block)
lappend extended_list [lindex $sorted_i 0]
for {{set i 1}} {{$i < $N}} {{incr i}} {{
    set x [lindex $sorted_i [expr $i - 1]]
    set y [lindex $sorted_i $i]
    set z [lindex $sorted_i [expr $i + 1]]
    if {{$y == [expr $x + 1] && $y == [expr $z - 1]}} {{
        lappend extended_list $y
    }} elseif {{$y == [expr $x + 1] && $y != [expr $z - 1]}} {{
        lappend extended_list $y
    }} elseif {{$y != [expr $x + 1] && $y == [expr $z - 1]}} {{
        lappend extended_list $y
    }} else {{
        # It's a singleton! Expand it.
        lappend extended_list [expr $y - 1]
        lappend extended_list $y
        lappend extended_list [expr $y + 1]
    }}
}}

# Write the raw extracted tunnel (Python will fix the TER cards next)
set b [atomselect top "residue $extended_list"]
$b writepdb {id_value}_tunnel_raw.pdb

exit
"""
    return tcl_script

In [17]:
import string

def fix_pdb_ter_chains_altloc(id_value):
    """Reads VMD PDB, removes alt-locs, inserts TER at gaps, and maps 2-letter chains."""
    input_file = f"{id_value}_tunnel_raw.pdb"
    output_file = f"{id_value}_tunnel_fix.pdb"
    valid_chains = string.ascii_uppercase + string.ascii_lowercase + string.digits
    chain_mapping = {}
    with open(input_file, 'r') as infile:
        lines = infile.readlines()
    with open(output_file, 'w') as outfile:
        prev_resid = None
        prev_original_chain = None
        for line in lines:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # --- 1. THE ALTLOC FILTER ---
                # Index 16 is the Alternate Location Indicator in standard PDBs
                alt_loc = line[16]
                # If it's B, C, 2, 3, etc., skip this line entirely!
                if alt_loc not in [' ', 'A', '1']:
                    continue 
                # If it's 'A' or '1', we keep it, but we MUST blank out the character
                # so the force field parser doesn't panic.
                line = line[:16] + ' ' + line[17:]
                # ----------------------------
                original_chain = line[20:22]
                try:
                    resid = int(line[22:26].strip())
                except ValueError:
                    resid = prev_resid 
                if original_chain not in chain_mapping:
                    new_id = valid_chains[len(chain_mapping) % len(valid_chains)]
                    chain_mapping[original_chain] = new_id
                current_new_chain = chain_mapping[original_chain]
                # --- 2. DETECT STRUCTURAL BREAKS ---
                is_break = False
                if prev_resid is not None:
                    if original_chain != prev_original_chain:
                        is_break = True
                    elif resid != prev_resid and resid != prev_resid + 1:
                        is_break = True
                if is_break:
                    outfile.write("TER\n")
                prev_resid = resid
                prev_original_chain = original_chain
                # --- 3. FIX THE FORMATTING ---
                new_line = line[:20] + " " + current_new_chain + line[22:]
                outfile.write(new_line)
            elif not line.startswith("END"):
                outfile.write(line)
        outfile.write("TER\nEND\n")
    print(f"Success! Purged alt-locs, mapped {len(chain_mapping)} chains, and inserted TER cards.")

In [18]:
# Specify the filename
filename = "tunnel_generator.tcl"

# Use a context manager to handle the file safely
with open(filename, 'w') as file:
    file.write(write_tcl_script(id))

In [19]:
%%capture
!vmd -dispdev text -e tunnel_generator.tcl

In [20]:
# Fixing the ChainID, TER/END and altloc problem
fix_pdb_ter_chains_altloc(id)

Success! Purged alt-locs, mapped 20 chains, and inserted TER cards.


In [21]:
# checking if the PDB file has some unmappable residues

In [22]:
def check_pdb_against_map(map_file, pdb_file):
    # 1. Parse the SMOG mapping file into a dictionary
    valid_map = {}
    with open(map_file, 'r') as f:
        for line in f:
            if line.startswith('residue'):
                parts = line.split()
                res_name = parts[1]
                # Store allowed atoms as a set for lightning-fast lookup
                allowed_atoms = set(parts[2:]) 
                valid_map[res_name] = allowed_atoms
    # 2. Parse the PDB file and track errors
    unrecognized_residues = set()
    unrecognized_atoms = set()
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # PDB standard fixed-width columns
                atom_name = line[12:16].strip()
                res_name = line[17:21].strip() # 17:21 handles 4-letter names like 'GLNT'
                # Check if the residue exists in the map
                if res_name not in valid_map:
                    unrecognized_residues.add(res_name)
                # Check if the atom exists for that specific residue
                elif atom_name not in valid_map[res_name]:
                    unrecognized_atoms.add((res_name, atom_name))
    # 3. Print the results clearly
    print("=== SMOG MAPPING CHECK RESULTS ===")
    if not unrecognized_residues and not unrecognized_atoms:
        print("✅ Success! All residues and atoms in the PDB exist in the mapping file.")
        return
    if unrecognized_residues:
        print("\n❌ UNRECOGNIZED RESIDUES (Not in map file):")
        for res in sorted(unrecognized_residues):
            print(f"   - {res}")
    if unrecognized_atoms:
        print("\n⚠️ UNRECOGNIZED ATOMS (Residue exists, but atom does not):")
        # Group atoms by residue for easier reading
        error_dict = {}
        for res, atom in unrecognized_atoms:
            error_dict.setdefault(res, []).append(atom)
        for res, atoms in sorted(error_dict.items()):
            print(f"   - {res}: {', '.join(sorted(atoms))}")

In [23]:
def fix_rna_nomenclature_and_psu(id_value):
    """Converts PSU to U, maps ring geometry, and fixes global RNA backbone nomenclature (' to *, OP1 to O1P)."""
    input_file = f"{id_value}_tunnel_fix.pdb"
    output_file = f"{id_value}_tunnel_fix_final.pdb"
    # The physical coordinate name-swap mapping for Pseudouridine
    psu_to_u_atoms = {
        "C5": "N1", "C4": "C2", "O4": "O2", 
        "C2": "C4", "O2": "O4", "N1": "C5"
    }
    psu_count = 0
    backbone_fix_count = 0
    with open(input_file, 'r') as infile:
        lines = infile.readlines()
    with open(output_file, 'w') as outfile:
        for line in lines:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # Extract the exact 4-character atom name block
                atom_block = line[12:16]
                # --- 1. GLOBAL BACKBONE NOMENCLATURE FIX ---
                modified_block = atom_block
                if "'" in modified_block:
                    modified_block = modified_block.replace("'", "*")
                if "OP1" in modified_block:
                    modified_block = modified_block.replace("OP1", "O1P")
                elif "OP2" in modified_block:
                    modified_block = modified_block.replace("OP2", "O2P")
                if modified_block != atom_block:
                    # Inject the fixed backbone name back in (keeps exact PDB spacing)
                    line = line[:12] + modified_block + line[16:]
                    backbone_fix_count += 1
                # --- 2. PSU TO U RING SWAP ---
                resname = line[17:20].strip()
                if resname == "PSU":
                    # Change residue name to U (Right-aligned to 3 characters)
                    line = line[:17] + "  U" + line[20:]
                    # Re-extract the stripped atom name in case it's one of our ring atoms
                    current_atom = line[12:16].strip()
                    if current_atom in psu_to_u_atoms:
                        new_atom = psu_to_u_atoms[current_atom]
                        formatted_atom = f" {new_atom:<3}"
                        line = line[:12] + formatted_atom + line[16:] 
                    psu_count += 1
            outfile.write(line)
    print(f"Success! Fixed {backbone_fix_count} backbone atoms (' to *, OP to O_P).")
    print(f"Converted and geometry-mapped {psu_count} PSU atoms. Saved as {output_file}.")

In [27]:
# Run it on your current ID
fix_rna_nomenclature_and_psu(id)

Success! Fixed 14828 backbone atoms (' to *, OP to O_P).
Converted and geometry-mapped 20 PSU atoms. Saved as 9qt5_tunnel_fix_final.pdb.


In [30]:
# Checks if the pdb file has consisten naming with the SMOG mapping
check_pdb_against_map('/mog-2.4.5/SBM_AA-amber-bonds_FME/map.SBM_AA-amber-bonds', id+'_tunnel_fix_final.pdb')

=== SMOG MAPPING CHECK RESULTS ===
✅ Success! All residues and atoms in the PDB exist in the mapping file.


## Inserting model of the NC and setting up SMOG

In [64]:
name = "9qt5"
item = 60 # change to 10, 20, 30, 40 if needed different lengths
smog_path = "~/soft/smog-2.4.5"

print(f"--- Processing {name} (Length {item}) ---")

# 1. Setup directories and files
!grep -v CRYST {name}_tunnel_fix_final.pdb > tmp && mv tmp {name}_tunnel_fix_final.pdb
!mkdir -p {item}
!cp {name}_tunnel_fix_final.pdb {item}/
!cp NC_FME_{item}.pdb {item}/

# 2. Build the combined PDB (Using paths instead of CD)
!grep -v "^END" {item}/NC_FME_{item}.pdb > {item}/tunnel_{item}.pdb
!cat {item}/{name}_tunnel_fix_final.pdb >> {item}/tunnel_{item}.pdb

# 3. Run SMOG Adjust
print("Running SMOG Adjust...")
!{smog_path}/bin/smog_adjustPDB -i {item}/tunnel_{item}.pdb -PDBresnum -o {item}/tunnel_{item}_adj.pdb -map {smog_path}/SBM_AA-amber-bonds_FME/map.SBM_AA-amber-bonds > /dev/null

# 4. Run SMOG2
print("Running SMOG2...")
!{smog_path}/bin/smog2 -i {item}/tunnel_{item}_adj.pdb -dname {item}/tunnel_{item} -t {smog_path}/SBM_AA-amber-bonds_FME/ > /dev/null

print("Done with SMOG.")

--- Processing 9qt5 (Length 60) ---
Running SMOG Adjust...
Running SMOG2...
Done with SMOG.


## GROMACS input files preparation and running energy minimization 

In [65]:
%%bash -s "$item"
# We pass the Python variable 'item' into bash as $1
length=$1

echo "--- Starting GROMACS pipeline for length: $length ---"

# 1. Clean the topology
# If the directory doesn't exist, this will now give a clear error
if [ -d "$length" ]; then
    cd $length
    echo "Cleaning topology..."
    sed -i -e '/\[ pairs \]/,/^\[ exclusions \]/{//!d}' tunnel_${length}.top
    sed -i -e '/\[ exclusions \]/,/^\[ system \]/{//!d}' tunnel_${length}.top
    echo "Setting up box..."
    ~/soft/gromacs-2021/bin/gmx editconf -f tunnel_${length}.gro -o tunnel_${length}_box.gro -c -d 1.0 -bt dodecahedron > /dev/null 2>&1
    echo "Preparing EM..."
    cp ../../../em.mdp .
    cp ../../../md.mdp .
    ~/soft/gromacs-2021/bin/gmx grompp -f em.mdp -c tunnel_${length}_box.gro -p tunnel_${length}.top -o em.tpr -maxwarn 1 > /dev/null 2>&1
    if [ -f "em.tpr" ]; then
        echo "Running Energy Minimization..."
        ~/soft/gromacs-2021/bin/gmx mdrun -deffnm em -c minim.pdb > /dev/null 2>&1
        echo "✅ EM Finished successfully."
    else
        echo "❌ ERROR: em.tpr was not created. Check your topology or .mdp file."
    fi
else
    echo "❌ ERROR: Directory '$length' does not exist. Did the SMOG step fail?"
fi

--- Starting GROMACS pipeline for length: 60 ---
Cleaning topology...
Setting up box...
Preparing EM...
Running Energy Minimization...
✅ EM Finished successfully.


In [66]:
# Define your mapping here: {item_length: (nc_end, nc_whole_end)}
atom_ranges = {
    10: (50, 58),
    20: (100, 108),
    30: (150, 158),
    40: (200, 208),
    60: (300, 308)
}

if item in atom_ranges:
    nc_end, nc_whole_end = atom_ranges[item]

print(f"Ranges for length {item}: NC (1-{nc_end}), NC_whole (1-{nc_whole_end})")

Ranges for length 60: NC (1-300), NC_whole (1-308)


In [67]:
%%bash -s "$item" "$nc_end" "$nc_whole_end"
# Access passed variables
length=$1
nc_stop=$2
whole_stop=$3

if [ -d "$length" ]; then
    cd $length
    echo "--- Creating Dynamic Index for length: $length ---"
    echo "Groups: NC (1-$nc_stop), NC_whole (1-$whole_stop)"
    # 1. Run make_ndx with dynamic ranges
    # We use the variables $nc_stop and $whole_stop inside the heredoc (EOF)
    ~/soft/gromacs-2021/bin/gmx make_ndx -f tunnel_${length}.pdb > /dev/null << EOF
a 1-$nc_stop
a 1-$whole_stop
0 &! 16
q
EOF
    # 2. Rename groups using the dynamic patterns
    # Note: We use double quotes in sed to allow variable expansion
    if [ -f "index.ndx" ]; then
        sed -i "s/System_&_!a_1-$nc_stop/RIBOSOME/g" index.ndx
        sed -i "s/a_1-$nc_stop/NC/g" index.ndx
        sed -i "s/a_1-$whole_stop/NC_whole/g" index.ndx
        echo "✅ Index file created and groups renamed."
    else
        echo "❌ ERROR: index.ndx was not generated!"
        exit 1
    fi
    # 3. Final grompp
    echo "Running final grompp..."
    [ ! -f "md.mdp" ] && cp ../md.mdp .
    ~/soft/gromacs-2021/bin/gmx grompp -f md.mdp -c minim.pdb -p tunnel_${length}.top -o md.tpr -n index.ndx -maxwarn 1 > /dev/null 2>&1
    if [ -f "md.tpr" ]; then
        echo "🚀 SUCCESS: md.tpr ready for length $length"
    else
        echo "❌ ERROR: grompp failed."
    fi
else
    echo "❌ ERROR: Directory '$length' not found."
fi

--- Creating Dynamic Index for length: 60 ---
Groups: NC (1-300), NC_whole (1-308)


                      :-) GROMACS - gmx make_ndx, 2021 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Teemu Mu

✅ Index file created and groups renamed.
Running final grompp...
🚀 SUCCESS: md.tpr ready for length 60


## Inserting new reference models of the NC 

In [ ]:
# To generate RNC models for the replicas

In [2]:
import numpy as np
import os

In [3]:
org = "pgin"
path = "/Projects/ribosome_tunnels/bacteria/"+org

In [4]:
os.chdir(path)

In [22]:
name = "9i5x"
item = 60
# Use os.path.expanduser to get the absolute path correctly
smog_path = os.path.expanduser("~/soft/smog-2.4.5")

print(f"--- Processing {name} (Length {item}) ---")

# 1. Setup
%cd {item}
!mkdir -p ref_1 ref_2

# 2. Build the combined PDBs properly
# For Ref 1
!grep -v "^END" first_ref.pdb > ref_1/tunnel_{item}.pdb
!echo "TER" >> ref_1/tunnel_{item}.pdb
!grep -v "^END" {name}_tunnel_fix_final.pdb >> ref_1/tunnel_{item}.pdb
!echo "END" >> ref_1/tunnel_{item}.pdb

# For Ref 2
!grep -v "^END" second_ref.pdb > ref_2/tunnel_{item}.pdb
!echo "TER" >> ref_2/tunnel_{item}.pdb
!grep -v "^END" {name}_tunnel_fix_final.pdb >> ref_2/tunnel_{item}.pdb
!echo "END" >> ref_2/tunnel_{item}.pdb

# 3. Run SMOG Adjust
print("Running SMOG Adjust...")
for ref in ["ref_1", "ref_2"]:
    input_pdb = f"{ref}/tunnel_{item}.pdb"
    output_pdb = f"{ref}/tunnel_{item}_adj.pdb"
    mapping = f"{smog_path}/SBM_AA-amber-bonds_FME/map.SBM_AA-amber-bonds"
    !{smog_path}/bin/smog_adjustPDB -i {input_pdb} -PDBresnum -o {output_pdb} -map {mapping} > /dev/null

# 4. Run SMOG2
print("Running SMOG2...")
for ref in ["ref_1", "ref_2"]:
    adj_pdb = f"{ref}/tunnel_{item}_adj.pdb"
    dname = f"{ref}/tunnel_{item}" # This sets the prefix for .top and .tpr
    template = f"{smog_path}/SBM_AA-amber-bonds_FME/"
    # Running SMOG2
    !{smog_path}/bin/smog2 -i {adj_pdb} -dname {dname} -t {template} > /dev/null

print(f"✅ Done with SMOG for item {item}.")
%cd ../

--- Processing 9i5x (Length 60) ---
/home/twlodarski/Projects/ribosome_tunnels/bacteria/pgin/60
Running SMOG Adjust...
Running SMOG2...
✅ Done with SMOG for item 60.
/home/twlodarski/Projects/ribosome_tunnels/bacteria/pgin


In [23]:
%%bash -s "$item"
length=$1
GMX_PATH="$HOME/soft/gromacs-2021/bin/gmx"

echo "--- Starting GROMACS pipeline for length: $length ---"

if [ -d "$length" ]; then
    cd "$length"
    # Iterate through both reference folders
    for ref in ref_1 ref_2; do
        if [ -d "$ref" ]; then
            echo "Processing $ref..."
            cd "$ref"
            # 1. Clean the topology
            # Note: We ensure we target tunnel_${length}.top inside this folder
            echo "  Cleaning topology..."
            sed -i -e '/\[ pairs \]/,/^\[ exclusions \]/{//!d}' tunnel_${length}.top
            sed -i -e '/\[ exclusions \]/,/^\[ system \]/{//!d}' tunnel_${length}.top
            # 2. Setting up box
            echo "  Setting up box..."
            $GMX_PATH editconf -f tunnel_${length}.gro -o tunnel_${length}_box.gro -c -d 1.0 -bt dodecahedron > /dev/null 2>&1
            # 3. Copy MDP files (Corrected path: they are now 3 levels up from here)
            cp ../../../../em.mdp .
            cp ../../../../md.mdp .
            # 4. Preparing EM
            echo "  Running grompp..."
            $GMX_PATH grompp -f em.mdp -c tunnel_${length}_box.gro -p tunnel_${length}.top -o em.tpr -maxwarn 1 > /dev/null 2>&1
            if [ -f "em.tpr" ]; then
                echo "  Running Energy Minimization..."
                $GMX_PATH mdrun -deffnm em -c minim.pdb > /dev/null 2>&1
                if [ -f "minim.pdb" ]; then
                    echo "  ✅ $ref: EM Finished successfully."
                else
                    echo "  ❌ $ref: ERROR: mdrun failed to produce minim.pdb."
                fi
            else
                echo "  ❌ $ref: ERROR: em.tpr not created. Check topology!"
            fi
            # Go back to length folder to process next ref
            cd ../
        else
            echo "  ⚠️ Skipping $ref: folder not found."
        fi
    done
    # Return to the base directory where the notebook is running
    cd ../
else
    echo "❌ ERROR: Directory '$length' does not exist."
fi

--- Starting GROMACS pipeline for length: 60 ---
Processing ref_1...
  Cleaning topology...
  Setting up box...
  Running grompp...
  Running Energy Minimization...
  ✅ ref_1: EM Finished successfully.
Processing ref_2...
  Cleaning topology...
  Setting up box...
  Running grompp...
  Running Energy Minimization...
  ✅ ref_2: EM Finished successfully.


In [24]:
# Define your mapping here: {item_length: (nc_end, nc_whole_end)}
atom_ranges = {
    10: (50, 58),
    20: (100, 108),
    30: (150, 158),
    40: (200, 208),
    60: (300, 308)
}

if item in atom_ranges:
    nc_end, nc_whole_end = atom_ranges[item]

print(f"Ranges for length {item}: NC (1-{nc_end}), NC_whole (1-{nc_whole_end})")

Ranges for length 60: NC (1-300), NC_whole (1-308)


In [25]:
%%bash -s "$item" "$nc_end" "$nc_whole_end"
length=$1
nc_stop=$2
whole_stop=$3
GMX_PATH="$HOME/soft/gromacs-2021/bin/gmx"

echo "--- Starting Dynamic Index & Grompp for length: $length ---"

if [ -d "$length" ]; then
    cd "$length"
    for ref in ref_1 ref_2; do
        if [ -d "$ref" ]; then
            echo "Processing $ref..."
            cd "$ref"
            # 1. Run make_ndx with dynamic ranges
            # We target the minim.pdb from the EM step to ensure coordinates match
            $GMX_PATH make_ndx -f minim.pdb -o index.ndx > /dev/null 2>&1 << EOF
a 1-$nc_stop
a 1-$whole_stop
0 &! 21
q
EOF
            # 2. Rename groups in index.ndx
            # GROMACS make_ndx usually turns 'a 1-48' into 'a_1-48'
            if [ -f "index.ndx" ]; then
                # Rename the groups to your standard names
                sed -i "s/a_1-$nc_stop/NC/g" index.ndx
                sed -i "s/a_1-$whole_stop/NC_whole/g" index.ndx
                # Find the complex 'System & ! NC' group. 
                # Note: GROMACS naming is literal, so we use a flexible sed pattern
                sed -i "s/System_&_!NC/RIBOSOME/g" index.ndx
                
                echo "  ✅ Index groups: NC (1-$nc_stop), NC_whole (1-$whole_stop) created."
            else
                echo "  ❌ ERROR: index.ndx not found in $ref."
                cd ../ && continue
            fi
            # 3. Final grompp for MD
            # Ensure we have the md.mdp (3 levels up)
            [ ! -f "md.mdp" ] && cp ../../../md.mdp .
            echo "  Running grompp for md.tpr..."
            $GMX_PATH grompp -f md.mdp -c minim.pdb -p tunnel_${length}.top -n index.ndx -o md.tpr -maxwarn 2 > /dev/null 2>&1
            if [ -f "md.tpr" ]; then
                echo "  🚀 SUCCESS: $ref/md.tpr is ready."
            else
                echo "  ❌ ERROR: grompp failed in $ref. Check for atom mismatches."
            fi
            cd ../
        fi
    done
    cd ../
else
    echo "❌ ERROR: Directory '$length' not found."
fi

--- Starting Dynamic Index & Grompp for length: 60 ---
Processing ref_1...
  ✅ Index groups: NC (1-300), NC_whole (1-308) created.
  Running grompp for md.tpr...
  🚀 SUCCESS: ref_1/md.tpr is ready.
Processing ref_2...
  ✅ Index groups: NC (1-300), NC_whole (1-308) created.
  Running grompp for md.tpr...
  🚀 SUCCESS: ref_2/md.tpr is ready.


In [1]:
# OK when the run is done we download all trajectories back and use Trajectory_analysis notebook to fix and fit all trajectories into one frame